<a href="https://colab.research.google.com/github/ppriyadarshini09/SparseAutoEncoders/blob/main/minigpt_sae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# Clone (or pull if already present) both repos
for repo, name in [
    ("https://github.com/ppriyadarshini09/miniGPT.git", "miniGPT"),
    ("https://github.com/ppriyadarshini09/SparseAutoEncoders.git", "SparseAutoEncoders"),
]:
    path = f"/content/{name}"
    if os.path.exists(path):
        !git -C {path} pull
    else:
        !git clone {repo} {path}

import sys
sys.path.insert(0, '/content/miniGPT')   # gives you CharTokenizer, GPT, etc.
sys.path.insert(0, '/content/SparseAutoEncoders')   # gives you your SAE class

import torch
import importlib
import copy
import train
import collect_activations
import train_sae
import inspect_features
importlib.reload(train)
importlib.reload(collect_activations)
importlib.reload(train_sae)
importlib.reload(inspect_features)

Mounted at /content/drive
Already up to date.
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 287 bytes | 287.00 KiB/s, done.
From https://github.com/ppriyadarshini09/SparseAutoEncoders
   79e86a4..f162dcc  main       -> origin/main
Updating 79e86a4..f162dcc
Fast-forward
 inspect_features.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
using device: cuda


<module 'inspect_features' from '/content/SparseAutoEncoders/inspect_features.py'>

In [2]:
# @title Train (Weak LLM)
cfg = copy.deepcopy(train.config)
cfg['run_name'] = 'weak_model'
cfg['block_size'] = 128
cfg['n_embed'] = 128
cfg['n_heads'] = 4
cfg['n_layers'] = 1
cfg['dropout'] = 0.1
cfg['batch_size'] = 64
cfg['max_steps'] = 3000
cfg['eval_every'] = 100

In [3]:
weak_model, tok = train.train(cfg)

Vocab size: 65
Train tokens: 1,003,854 Val tokens: 111,540
GPT initialized - 0.222848M parameters
BASE_DIR: /content/miniGPT
checkpoints will be stored: /content/miniGPT/checkpoints
step     0 | train loss: 4.2327 | val loss: 4.2290
 -> saved checkpoint (val loss 4.2290)
step   100 | train loss: 2.6115 | val loss: 2.6187
 -> saved checkpoint (val loss 2.6187)
step   200 | train loss: 2.4967 | val loss: 2.4980
 -> saved checkpoint (val loss 2.4980)
step   300 | train loss: 2.4588 | val loss: 2.4616
 -> saved checkpoint (val loss 2.4616)
step   400 | train loss: 2.4175 | val loss: 2.4368
 -> saved checkpoint (val loss 2.4368)
step   500 | train loss: 2.3798 | val loss: 2.3989
 -> saved checkpoint (val loss 2.3989)
step   600 | train loss: 2.3163 | val loss: 2.3368
 -> saved checkpoint (val loss 2.3368)
step   700 | train loss: 2.2338 | val loss: 2.2752
 -> saved checkpoint (val loss 2.2752)
step   800 | train loss: 2.1890 | val loss: 2.2362
 -> saved checkpoint (val loss 2.2362)
step   9

In [4]:
# Inference: Generate text with starting seed
from train import generate_sample
prompt = "To be"
print(generate_sample(weak_model, tok, prompt))

To be the stall king when that,
What then somes word: the be me my to him.

CAMILLO:
My slord dis upon a for land men!
The they now thang, muts
Of the do conne.

That han seet you this entle thee heart par


In [5]:
!ls -lh miniGPT

total 104K
-rw-r--r-- 1 root root 4.4K Aug  2 23:53 attention.py
drwxr-xr-x 2 root root 4.0K Aug  2 23:53 checkpoints
drwxr-xr-x 2 root root 4.0K Aug  2 23:53 data
-rw-r--r-- 1 root root  11K Aug  2 23:53 model.py
drwxr-xr-x 2 root root 4.0K Aug  2 23:53 __pycache__
-rw-r--r-- 1 root root  19K Aug  2 23:53 README.md
-rw-r--r-- 1 root root 3.2K Aug  2 23:53 test_attention.py
-rw-r--r-- 1 root root 5.4K Aug  2 23:53 test_block.py
-rw-r--r-- 1 root root 1.4K Aug  2 23:53 test_bpe.py
-rw-r--r-- 1 root root 4.6K Aug  2 23:53 test_embeddings.py
-rw-r--r-- 1 root root 5.7K Aug  2 23:53 test_gpt.py
-rw-r--r-- 1 root root 3.4K Aug  2 23:53 test_multihead.py
-rw-r--r-- 1 root root 1.1K Aug  2 23:53 test_tokenizer.py
-rw-r--r-- 1 root root 3.5K Aug  2 23:53 tokenizer.py
-rw-r--r-- 1 root root 5.5K Aug  2 23:53 train.py


In [6]:
!mkdir -p /content/drive/MyDrive/LLMs/miniGPT/checkpoints
!cp -r /content/miniGPT/checkpoints/* /content/drive/MyDrive/LLMs/miniGPT/checkpoints/
!mkdir -p /content/drive/MyDrive/LLMs/miniGPT/data
!cp -r /content/miniGPT/data/* /content/drive/MyDrive/LLMs/miniGPT/data/
!mkdir -p /content/drive/MyDrive/LLMs/SparseAutoEncoders/activations/
!mkdir -p /content/drive/MyDrive/LLMs/SparseAutoEncoders/checkpoints/

In [7]:
!ls /content/drive/MyDrive/LLMs/miniGPT/data

shakespeare.txt


In [8]:
# @title Save checkpoints for SAE
ckpt_path = '/content/drive/MyDrive/LLMs/miniGPT/checkpoints/weak_model_best_model.pt'
data_path = '/content/drive/MyDrive/LLMs/miniGPT/data/shakespeare.txt'

out_path = '/content/drive/MyDrive/LLMs/SparseAutoEncoders/activations/weak_model_shakespeare.pt'

collect_activations.save_activations(
    ckpt_path=ckpt_path,
    data_path=data_path,
    layer_idx=-1, # last transformer layer
    max_tokens=200000,
    batch_size=256,
    out_path=out_path
)

GPT initialized - 0.222848M parameters
Loaded checkpoint (step 2900, val_loss 1.887623)
Config: n_layers=1 n_embed=128 block_size=128 -> d_mlp=512
token stream shape: (1115394,)
Total token available: 1,115,394
Total 1562 chunks of 128 tokens each
Finally collecting activation for 199936 positions
# of chunks to process: 1562
Processing chunk 0
    processed 256/1562 chunks
Processing chunk 256
Processing chunk 512
Processing chunk 768
Processing chunk 1024
Processing chunk 1280
Processing chunk 1536

Collected activations: (199936, 512)  (N token-positions x d_mlp)
Mean: 0.0271  Std: 0.3244  Min: -0.1700  Max: 4.5289
Note: these raw GELU activations are NOT sparse (GELU rarely outputs exactly 0) -> that's expected. The SAE's job later is to impose sparsity on top of this dense signal.
Saved activations to /content/drive/MyDrive/LLMs/SparseAutoEncoders/activations/weak_model_shakespeare.pt


In [9]:
train_sae.train(activations_path=out_path,
      save_path='/content/drive/MyDrive/LLMs/SparseAutoEncoders/checkpoints/weak_gpt.pt'
      )

Expanding 512 -> 2048 features
Loaded activations: (199936, 512)
Loaded metadata: {'positions': tensor([     0,      1,      2,  ..., 199933, 199934, 199935], device='cuda:0'), 'layer_idx': -1, 'config': {'run_name': 'weak_model', 'data_path': 'data/shakespeare.txt', 'ckpt_dir': 'checkpoints', 'vocab_size': 65, 'block_size': 128, 'n_embed': 128, 'n_heads': 4, 'n_layers': 1, 'dropout': 0.1, 'batch_size': 64, 'max_steps': 3000, 'eval_every': 100, 'lr': 0.0003, 'grad_clip': 1.0, 'train_split': 0.9}, 'ckpt_path': '/content/drive/MyDrive/LLMs/miniGPT/checkpoints/weak_model_best_model.pt'}
Epoch: 0
Epoch: 0, Batch: 0, Loss: 0.30388393998146057
Epoch: 0, Batch: 100, Loss: 0.09023039788007736
Epoch: 10
Epoch: 20
Epoch: 30
Epoch: 40


In [11]:
ckpt = torch.load('/content/drive/MyDrive/LLMs/SparseAutoEncoders/checkpoints/weak_gpt.pt', map_location='cpu', weights_only=False)
print(ckpt['metrics'])

{'recon_loss': 0.06618484854698181, 'l1_loss': 4.688881874084473, 'l0': 17.57421875, 'dead_frac': 1015.212890625}


In [26]:
fs = inspect_features.FeatureStats(
    data_path='/content/drive/MyDrive/LLMs/miniGPT/data/shakespeare.txt',
    saved_act_path='/content/drive/MyDrive/LLMs/SparseAutoEncoders/activations/weak_model_shakespeare.pt',
    saved_sae_path='/content/drive/MyDrive/LLMs/SparseAutoEncoders/checkpoints/weak_gpt.pt'
)

Loaded SAE: d_in=512, d_hidden=2048


In [27]:
fs.summarize_feature_stats()


==== Corpus wide feature stats ====
Total features: 2048
Dead features: 0
Very dense features (fires on >50% of tokens, likely uninterpretable): 0 (0.0%)
Average L0 (features active per token): 17.415 / 2048


In [28]:
fs.show_feature_density(9)
fs.show_top_activating_examples(9)


==== Feature 9 : density stats ====
Fires on 23516/199936 tokens (0.1176 of all positions)
When active -- mean: 0.400 max: 2.165 min: 0.000

==== Feature 9: 10 activating examples ====
Rank  1, activation=2.165 ...ou have been ere now, and what you are;\n[[W]] ithal, what I have been, and what I am.
Rank  2, activation=2.143 ...,\nAnd presently repair to Crosby Place;\n[[W]] here, after I have solemnly interr'd\nAt
Rank  3, activation=2.136 ...roat to thee and to thy ancient malice;\n[[W]] hich not to cut would show thee but a f
Rank  4, activation=2.134 ...threshold. Why, thou Mars! I tell thee,\n[[W]] e have a power on foot; and I had purpo
Rank  5, activation=2.122 ...ud to do't.\n\nBRUTUS:\nI heard him swear,\n[[W]] ere he to stand for consul, never would
Rank  6, activation=2.116 ...should find you lions, finds you hares;\n[[W]] here foxes, geese: you are no surer, no
Rank  7, activation=2.108 ... dark spirit, in 's nervy arm doth lie;\n[[W]] hich, being advanced, declines, and t

In [30]:
fs.show_feature_density(111)
fs.show_top_activating_examples(111)


==== Feature 111 : density stats ====
Fires on 4804/199936 tokens (0.0240 of all positions)
When active -- mean: 0.172 max: 0.581 min: 0.000

==== Feature 111: 10 activating examples ====
Rank  1, activation=0.581 ...\nMENENIUS:\nBecause you talk of pride now[[,]] --will you not be angry?\n\nBoth:\nWell, w
Rank  2, activation=0.552 ...hall'!\nO good but most unwise patricians[[!]]  why,\nYou grave but reckless senators, 
Rank  3, activation=0.540 ...e their provand\nOnly for bearing burdens[[,]]  and sore blows\nFor sinking under them.
Rank  4, activation=0.532 ...se your aid\nIn this so never-needed help[[,]]  yet do not\nUpbraid's with our distress
Rank  5, activation=0.531 ...UMNIA:\nShould we be silent and not speak[[,]]  our raiment\nAnd state of bodies would 
Rank  6, activation=0.530 ...hou mayst be damned for that wicked deed[[!]] \nO, he was gentle, mild, and virtuous!\n
Rank  7, activation=0.520 ...ow, as I live, I will. My nobler friends[[,]] \nI crave their pardons:\nFor t

In [ ]:
# @title TRAIN (Default Config)
cfg = copy.deepcopy(config)
cfg['run_name'] = 'default'
cfg['max_steps'] = 2500

model, tok = train(cfg)

Train tokens: 1,003,854 Val tokens: 111,540
GPT initialized - 10.763904M parameters
step     0 | train loss: 4.2237 | val loss: 4.2276
 -> saved checkpoint (val loss 4.2276)
step   250 | train loss: 2.2293 | val loss: 2.2649
 -> saved checkpoint (val loss 2.2649)
step   500 | train loss: 1.7249 | val loss: 1.8707
 -> saved checkpoint (val loss 1.8707)
step   750 | train loss: 1.4913 | val loss: 1.6741
 -> saved checkpoint (val loss 1.6741)
step  1000 | train loss: 1.3665 | val loss: 1.5642
 -> saved checkpoint (val loss 1.5642)
step  1250 | train loss: 1.2846 | val loss: 1.5315
 -> saved checkpoint (val loss 1.5315)
step  1500 | train loss: 1.2266 | val loss: 1.4983
 -> saved checkpoint (val loss 1.4983)
step  1750 | train loss: 1.1672 | val loss: 1.4945
 -> saved checkpoint (val loss 1.4945)
step  2000 | train loss: 1.1214 | val loss: 1.4649
 -> saved checkpoint (val loss 1.4649)
step  2250 | train loss: 1.0900 | val loss: 1.4774

Training complete. Best val loss: 1.4649


In [ ]:
from train import generate_sample
prompt = "To be"
print(generate_sample(model, tok, prompt))

To be past and too much a lead-horse
To be consul, if I am come.

LUCIO:
We were in mine own strength.

ISABELLA:
Ay, in Lord Angelo, and Juliet lies,
Or so that hated us that seems rancording
Be the fresh


In [ ]:
prompt = "romeo:"
print(generate_sample(model, tok, prompt))

romeo:
I cry, that thou livest me, since I wish thy complaint,
I'll mouth you, you can be not consul, my chamber
The matter shall grow time and usurp my lusters.

DUKE OF AUMERLE:
I know not thee.

DUKE OF 


In [ ]:
for name, param in model.named_parameters():
  print(f"{name} shape: {param.shape}")

transformer.embedding.token_embed.embedding.weight shape: torch.Size([65, 384])
transformer.embedding.pos_embed.embedding.weight shape: torch.Size([256, 384])
transformer.blocks.0.ln1.weight shape: torch.Size([384])
transformer.blocks.0.ln1.bias shape: torch.Size([384])
transformer.blocks.0.attn.heads.0.query.weight shape: torch.Size([64, 384])
transformer.blocks.0.attn.heads.0.key.weight shape: torch.Size([64, 384])
transformer.blocks.0.attn.heads.0.value.weight shape: torch.Size([64, 384])
transformer.blocks.0.attn.heads.1.query.weight shape: torch.Size([64, 384])
transformer.blocks.0.attn.heads.1.key.weight shape: torch.Size([64, 384])
transformer.blocks.0.attn.heads.1.value.weight shape: torch.Size([64, 384])
transformer.blocks.0.attn.heads.2.query.weight shape: torch.Size([64, 384])
transformer.blocks.0.attn.heads.2.key.weight shape: torch.Size([64, 384])
transformer.blocks.0.attn.heads.2.value.weight shape: torch.Size([64, 384])
transformer.blocks.0.attn.heads.3.query.weight shap